In [46]:
import pandas as pd
import numpy as np

In [47]:
df = pd.read_csv(r'D:\Data Analysis\data analysis intern\Task2\Data_Files\Raw_Files\FactSale.csv')

In [48]:
df

,Sale Key,City Key,Customer Key,Bill To Customer Key,Stock Item Key,Invoice Date Key,Delivery Date Key,Salesperson Key,WWI Invoice ID,Description,...,Quantity,Unit Price,Tax Rate,Total Excluding Tax,Tax Amount,Profit,Total Including Tax,Total Dry Items,Total Chiller Items,Lineage Key
0,49265,41568,0,0,204,10/22/2013,10/23/2013,83,15189,DBA joke mug - mind if I join you? (White),...,6,13.0,15,78.0,11.7,51.0,89.7,6,0,11
1,49372,48937,0,0,173,10/22/2013,10/23/2013,83,15218,Developer joke mug - a foo walks into a bar (B...,...,6,13.0,15,78.0,11.7,51.0,89.7,6,0,11
2,50379,91464,0,0,174,10/28/2013,10/29/2013,70,15525,Developer joke mug - a foo walks into a bar (W...,...,6,13.0,15,78.0,11.7,51.0,89.7,6,0,11
3,51852,48937,0,0,171,11/6/2013,11/7/2013,74,15986,Developer joke mug - this code was generated b...,...,6,13.0,15,78.0,11.7,51.0,89.7,6,0,11
4,52374,89925,0,0,203,11/8/2013,11/9/2013,39,16138,DBA joke mug - mind if I join you? (Black),...,6,13.0,15,78.0,11.7,51.0,89.7,6,0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26392,219912,108988,273,202,220,4/21/2016,4/22/2016,165,67945,Novelty chilli chocolates 500g,...,72,14.5,10,1044.0,104.4,414.0,1148.4,0,72,11
26393,220186,112229,0,0,220,4/22/2016,4/23/2016,171,68033,Novelty chilli chocolates 500g,...,12,14.5,10,174.0,17.4,69.0,191.4,0,12,11
26394,223515,112381,0,0,220,5/9/2016,5/10/2016,161,69055,Novelty chilli chocolates 500g,...,96,14.5,10,1392.0,139.2,552.0,1531.2,0,96,11
26395,224963,72760,23,1,220,5/14/2016,5/15/2016,170,69493,Novelty chilli chocolates 500g,...,12,14.5,10,174.0,17.4,69.0,191.4,0,12,11


# Exploratory Data Anlaysis 

In [49]:
df.info()

# 1- change Invoice Date Key , Delivery Date Key from object to date

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26397 entries, 0 to 26396
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Sale Key              26397 non-null  int64  
 1   City Key              26397 non-null  int64  
 2   Customer Key          26397 non-null  int64  
 3   Bill To Customer Key  26397 non-null  int64  
 4   Stock Item Key        26397 non-null  int64  
 5   Invoice Date Key      26397 non-null  object 
 6   Delivery Date Key     26384 non-null  object 
 7   Salesperson Key       26397 non-null  int64  
 8   WWI Invoice ID        26397 non-null  int64  
 9   Description           26397 non-null  object 
 10  Package               26397 non-null  object 
 11  Quantity              26397 non-null  int64  
 12  Unit Price            26397 non-null  float64
 13  Tax Rate              26397 non-null  int64  
 14  Total Excluding Tax   26397 non-null  float64
 15  Tax Amount         

In [22]:
# Check for exact row-level duplicates
row_duplicates = df.duplicated().sum()

# Check for duplicate primary keys
pk_duplicates = df["Sale Key"].duplicated().sum()

print(f"Full-row duplicates: {row_duplicates}")
print(f"Duplicate Sale Keys: {pk_duplicates}")

Full-row duplicates: 0
Duplicate Sale Keys: 0


# Data Cleaning and Validation for Analysis

In [23]:
# Clean string columns (remove the spaces)
text_cols = ["Description", "Package"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

print(f"Unique Package types: {df['Package'].unique().tolist()}")

Unique Package types: ['Each', 'Packet', 'Pair', 'Bag']


In [24]:
# Parse date strings into pandas datetime (change Invoice Date Key , Delivery Date Key from object to date)
df["Invoice Date Key"] = pd.to_datetime(df["Invoice Date Key"], format="%m/%d/%Y")  # date formation month/day/year

df["Delivery Date Key"] = pd.to_datetime(df["Delivery Date Key"], format="%m/%d/%Y")  # date formation month/day/year

print("Invoice Date dtype:", df["Invoice Date Key"].dtype)
print("Delivery Date dtype:", df["Delivery Date Key"].dtype)

Invoice Date dtype: datetime64[ns]
Delivery Date dtype: datetime64[ns]


In [25]:
# Inspect the invoice date for missing delivery records
missing_delivery_invoices = df[df["Delivery Date Key"].isna()]["Invoice Date Key"].unique() # orders made but not delivered

print(f"Invoice date(s) for missing delivery dates: {missing_delivery_invoices}")  

# Add Delivery Status flag column 
df["Delivery Status"] = np.where(df["Delivery Date Key"].isna(), "Pending", "Delivered")

# Verify flag counts
print("\nDelivery Status Breakdown:")
print(df["Delivery Status"].value_counts())

Invoice date(s) for missing delivery dates: <DatetimeArray>
['2016-05-31 00:00:00']
Length: 1, dtype: datetime64[ns]

Delivery Status Breakdown:
Delivery Status
Delivered    26384
Pending         13
Name: count, dtype: int64


In [26]:
# Calculate percentage of zero customer key rows
zero_customer_count = (df["Customer Key"] == 0).sum()
zero_customer_pct = (zero_customer_count / len(df)) * 100


print(f"Customer Key = 0 rows: {zero_customer_count} ({zero_customer_pct:.2f}%)")

# Create a customer classification label
df["Customer Label"] = np.where(df["Customer Key"] == 0, "Unknown/Walk-in Customer", df["Customer Key"].astype(str))

Customer Key = 0 rows: 9077 (34.39%)


In [27]:
# Check uniqueness of Lineage Key
lineage_unique_values = df["Lineage Key"].nunique()
print(f"Lineage Key unique values count: {lineage_unique_values}")

# Drop constant ETL column as all the values is 11 
df = df.drop(columns=["Lineage Key"])
print("Lineage Key removed successfully.")

Lineage Key unique values count: 1
Lineage Key removed successfully.


In [28]:
# 1- Check Total Excluding Tax
Total_Excluding_Tax_calc = df["Quantity"] * df["Unit Price"]
mismatches = (np.abs(df["Total Excluding Tax"] - Total_Excluding_Tax_calc) > 1e-5).sum()

if mismatches == 0:
    print(
        f"✅ Total Excluding Tax: PASS (0 mismatches out of {len(df):,} rows)"
    )
else:
    print(f"❌ Total Excluding Tax ERROR: {mismatches:,} mismatches found!")

✅ Total Excluding Tax: PASS (0 mismatches out of 26,397 rows)


In [29]:
# 2- Check Tax Amount & Rounding Logic
from decimal import ROUND_HALF_UP, Decimal


def round_half_up(val, decimals=2):
    return float(
        Decimal(str(val)).quantize(
            Decimal("1." + "0" * decimals), rounding=ROUND_HALF_UP
        )
    )


# Calculate expected tax locally
calc_tax_unrounded = df["Total Excluding Tax"] * df["Tax Rate"] / 100.0
calc_tax_halfup = calc_tax_unrounded.apply(lambda x: round_half_up(x, 2))

tax_mismatches = (np.round(df["Tax Amount"] - calc_tax_halfup, 2) != 0).sum()

if tax_mismatches == 0:
    print(
        f"✅ Tax Amount: PASS (0 mismatches using Round-Half-Up out of {len(df):,} rows)"
    )
else:
    print(f"❌ Tax Amount ERROR: {tax_mismatches:,} mismatches found!")

✅ Tax Amount: PASS (0 mismatches using Round-Half-Up out of 26,397 rows)


In [30]:
# 3- Check Total Including Tax
Total_Including_Tax_calc = df["Total Excluding Tax"] + df["Tax Amount"]
mismatches = (np.abs(df["Total Including Tax"] - Total_Including_Tax_calc) > 1e-5).sum()

if mismatches == 0:
    print(
        f"✅ Total Including Tax: PASS (0 mismatches out of {len(df):,} rows)"
    )
else:
    print(
        f"❌ Total Including Tax ERROR: {mismatches:,} mismatches found!"
    )

✅ Total Including Tax: PASS (0 mismatches out of 26,397 rows)


In [31]:
# 4- Check Item Quantity Consistency
Quantity_calc = df["Total Dry Items"] + df["Total Chiller Items"]
mismatches = (df["Quantity"] != Quantity_calc).sum()

if mismatches == 0:
    print(f"✅ Quantity Balance: PASS (0 mismatches out of {len(df):,} rows)")
else:
    print(f"❌ Quantity Balance ERROR: {mismatches:,} mismatches found!")

✅ Quantity Balance: PASS (0 mismatches out of 26,397 rows)


In [32]:
# 6- Check Negative Profit Pattern
loss_mask = df["Profit"] < 0
loss_count = loss_mask.sum()
loss_pct = (loss_count / len(df)) * 100

loss_margins = (df.loc[loss_mask, "Profit"] / df.loc[loss_mask, "Total Excluding Tax"]).round(4)

print(f"🔍 Negative Profit Count: {loss_count:,} rows ({loss_pct:.2f}% of total)")
print("\nMost Frequent Negative Profit Margins:")
print(loss_margins.value_counts().head())

🔍 Negative Profit Count: 566 rows (2.14% of total)

Most Frequent Negative Profit Margins:
-0.0556    492
-1.5000     31
-2.7500     20
-1.4583      7
-2.8000      6
Name: count, dtype: int64


In [33]:
# 7- Check Delivery Lag Rules
deliv_mask = df["Delivery Date Key"].notna()
deliv_lags = (df.loc[deliv_mask, "Delivery Date Key"]- df.loc[deliv_mask, "Invoice Date Key"]).dt.days

unique_lags = deliv_lags.unique().tolist()

print(f"Delivery Lag Values (in days): {unique_lags}")
if unique_lags == [1]:
    print("✅ Delivery Lag Rule: PASS (Strict next-day delivery on all rows)")
else:
    print(f"⚠️ Delivery Lag Alert: Found unexpected lags: {unique_lags}")

Delivery Lag Values (in days): [1]
✅ Delivery Lag Rule: PASS (Strict next-day delivery on all rows)


In [34]:
# 8- Per-Product Outlier Scan
def detect_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return (series < (q1 - 1.5 * iqr)) | (series > (q3 + 1.5 * iqr))


price_outliers_count = (df.groupby("Stock Item Key")["Unit Price"].transform(detect_iqr_outliers).sum())
qty_outliers_count = ( df.groupby("Stock Item Key")["Quantity"].transform(detect_iqr_outliers).sum())


print(f"🔍 Per-Product Unit Price Outliers: {price_outliers_count} rows ({price_outliers_count/len(df):.2%})")
print(f"🔍 Per-Product Quantity Outliers: {qty_outliers_count} rows ({qty_outliers_count/len(df):.2%})")

🔍 Per-Product Unit Price Outliers: 74 rows (0.28%)
🔍 Per-Product Quantity Outliers: 4 rows (0.02%)


# Feature Engineering 
add some columns for Analysis 

In [35]:
# 1- Date features
df["Invoice Year"] = df["Invoice Date Key"].dt.year
df["Quarter"] = df["Invoice Date Key"].dt.quarter
df["Month"] = df["Invoice Date Key"].dt.month
df["Month Name"] = df["Invoice Date Key"].dt.strftime("%B")
df["Day of Week"] = df["Invoice Date Key"].dt.day_name()

In [36]:
# 2- Profit Margin 
df["Profit Margin %"] = df["Profit"] / df["Total Excluding Tax"]

In [37]:
# 3- Loss flags
df["Is_Loss_Sale"] = df["Profit"] < 0

In [38]:
# 4- Order Size bucket
def order_size(qty):
    if qty <= 10:
        return "Small"
    elif qty <= 60:
        return "Medium"
    else:
        return "Large"

df["Order Size"] = df["Quantity"].apply(order_size)

In [39]:
# # 5- Item Type (Dry vs Chiller)
# df["Item Type"] = np.where(df["Total Chiller Items"] > 0, "Chiller", "Dry")

In [40]:
# 6- Customer Type
df["Customer Type"] = np.where(df["Customer Key"] == 0, "Unknown/Walk-in", "Known")

In [41]:
# 7- Product Category (keyword matching)
def classify_product_category(description):
    if pd.isna(description):
        return "Uncategorized"

    text = str(description).lower()

    if any(kw in text for kw in ["mug", "cup", "bottle", "glass", "joke mug"]):
        return "Drinkware & Novelty"
    elif any(
        kw in text
        for kw in [
            "t-shirt",
            "shirt",
            "hoodie",
            "beanie",
            "cap",
            "jacket",
            "slipper",
            "glove",
        ]
    ):
        return "Apparel & Wearables"
    elif any(kw in text for kw in ["usb", "flash drive", "cable", "adapter"]):
        return "Electronics & Accessories"
    elif any(
        kw in text
        for kw in [
            "wrap",
            "carton",
            "bag",
            "box",
            "tape",
            "machine",
            "cushion",
            "void fill",
        ]
    ):
        return "Packaging & Shipping Supplies"
    elif any(kw in text for kw in ["toy", "car", "mask", "rc"]):
        return "Toys & Seasonal"
    else:
        return "General Merchandise"


df["Product_Category"] = df["Description"].apply(classify_product_category)

In [42]:
df.to_csv(r'D:\Data Analysis\data analysis intern\Task2\Data_Files\Cleaned_files\cleaned_sales.csv')

# Key Insights

1- Revenue grew every full year from 2013 to 2015 ($5.26M → $5.84M → $6.35M, +20.7% cumulative), but the 5 completed months of 2016 ($2.43M) annualize to roughly $5.83M — below 2015's pace.

2- Packaging & Shipping Supplies alone drives 58.1% of total revenue ($11.5M) — nearly 6x the next-largest category, Apparel & Footwear (21.3%).

3- "Large" orders (61+ units) are only 23.8% of all line items but generate 49.8% of total revenue — essentially half the business rides on a quarter of the transactions.

4- Every fulfilled order (26,384 of 26,397) shipped exactly one day after invoicing, with zero exceptions across 3+ years — the only 13 unfulfilled orders were all dated the very last day in the dataset.